In [1]:
import pandas as pd
import numpy as np

# 1. Load Datasets
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
translations = pd.read_csv('../data/raw/product_category_name_translation.csv')

# 2. Merge Product Category Translations
products = products.merge(translations, on='product_category_name', how='left')
products.drop(columns=['product_category_name'], inplace=True)
products.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)
products['product_category'] = products['product_category'].fillna('Unknown')

# 3. Convert Order Timestamps to Datetime
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 4. Filter for Delivered Orders
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# 5. Create Master Sales Table
master_df = order_items.merge(orders_clean, on='order_id', how='inner')
master_df = master_df.merge(customers, on='customer_id', how='left')
master_df = master_df.merge(products[['product_id', 'product_category']], on='product_id', how='left')

# Calculate Total Order Item Value
master_df['total_value'] = master_df['price'] + master_df['freight_value']

# 6. Export Cleaned Dataset to data/processed
master_df.to_csv('../data/processed/cleaned_master_sales.csv', index=False)
print("Data Cleaning Complete! Saved to data/processed/cleaned_master_sales.csv")

Data Cleaning Complete! Saved to data/processed/cleaned_master_sales.csv
